In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
 
import shutil
import numpy as np
import tensorflow as tf
 
from keras.models import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

In [ ]:
train_dir = "archive/train"
test_dir  = "archive/test"

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
)
test_datagen = ImageDataGenerator(rescale=1./255)
 
training_set = train_datagen.flow_from_directory(
    train_dir, target_size=(64, 64), batch_size=32, class_mode="binary"
)
testing_set = test_datagen.flow_from_directory(
    test_dir, target_size=(64, 64), batch_size=32, class_mode="binary"
)

print("class_indices:", training_set.class_indices)
assert len(training_set.class_indices) == 2, \
    "Expected 2 classes! Your data is not organized into cats/ and dogs/ subfolders."

In [ ]:
cnn_model = Sequential([
    Input(shape=(64, 64, 3)),
 
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
 
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
 
    Flatten(),
    Dense(units=128, activation="relu"),
    Dropout(0.5),
    Dense(units=1, activation="sigmoid"),
])
 
cnn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [ ]:
callbacks = [
    CSVLogger("log.csv", append=False, separator=";"),
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_model.keras", monitor="val_accuracy", save_best_only=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

In [ ]:
history = cnn_model.fit(
    training_set,
    epochs=5,
    validation_data=testing_set,
    callbacks=callbacks,
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history["accuracy"], label="train")
ax1.plot(history.history["val_accuracy"], label="val")
ax1.set_title("Accuracy"); ax1.set_xlabel("epoch"); ax1.legend()
ax2.plot(history.history["loss"], label="train")
ax2.plot(history.history["val_loss"], label="val")
ax2.set_title("Loss"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()